# 🚀 md-editor 端侧专属小模型: Google Colab L4/T4 矩阵微调与发布

本 Notebook 可在 **Google Colab (推荐 L4 / T4 GPU)** 上一键完成：
1. **环境自检**：检测 NVIDIA L4 / T4 GPU 与硬件加速状态
2. **多模型矩阵微调**：支持单独训练 **0.5B Lite** / **1.5B Standard**，或 **一键全矩阵全自动打包**
3. **自动量化与增量聚合**：转换为 `Q4_K_M` GGUF，并自动增量合并到同一个 `manifest.json`
4. **自动发布**：一键推送到 GitHub Releases 同一版本 Tag 下

In [ ]:
#@title ⚙️ [1/4] 配置训练参数与基座选择
#@markdown 请在右侧面板选择你要微调的模型规格：

model_tier = "0.5B (Lite - L4约5分钟)" #@param ["0.5B (Lite - L4约5分钟)", "1.5B (Standard - L4约15分钟)", "All Matrix (0.5B + 1.5B 全矩阵打包 - 约20分钟)"]
version_tag = "v1.0.0" #@param {type:"string"}

if "1.5B" in model_tier and "All" not in model_tier:
    BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
else:
    BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"🎯 已选择模式: {model_tier}")
print(f"🏷️ 版本标签:   {version_tag}")

# 验证 GPU 状态 (需显示 L4 / T4)
!nvidia-smi

In [ ]:
#@title 📦 [2/4] 克隆/更新仓库并预装全套依赖环境
import os
if not os.path.exists('/content/md-editor-models'):
    !git clone https://github.com/wmasfoe/md-editor-models.git /content/md-editor-models
else:
    !git -C /content/md-editor-models pull origin master

%cd /content/md-editor-models
!git pull origin master
!pip install -q trl peft pangu datasets transformers accelerate sentencepiece gguf protobuf "torchao>=0.16.0"

In [ ]:
#@title 🔑 [3/4] 配置 GitHub Token (用于自动发布 Release)
import os
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None

if not token and not os.environ.get('GH_TOKEN') and not os.environ.get('GITHUB_TOKEN'):
    token = input("请输入你的 GitHub Token (按回车直接上传): ").strip()

if token:
    os.environ['GH_TOKEN'] = token
    os.environ['GITHUB_TOKEN'] = token
    print("✅ GitHub Token 配置成功！")
else:
    print("ℹ️ 未提供 Token，训练完成后模型将保存在 output 目录。")

In [ ]:
#@title 🚀 [4/4] 启动一键微调、量化与多模型聚合发布！
!git pull origin master
!chmod +x scripts/release_model.sh

if "All" in model_tier:
    print("🔥 [1/2] 正在微调与发布 0.5B Lite 模型...")
    !./scripts/release_model.sh $version_tag Qwen/Qwen2.5-0.5B-Instruct
    print("\n🔥 [2/2] 正在微调与发布 1.5B Standard 模型...")
    !./scripts/release_model.sh $version_tag Qwen/Qwen2.5-Coder-1.5B-Instruct
else:
    !./scripts/release_model.sh $version_tag $BASE_MODEL

print("\n🎉 全流程执行完毕！多模型矩阵已上线 GitHub Releases！")